In [ ]:
import torch, sys, os

print(f'Python:  {sys.version[:6]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available')

In [ ]:
!pip install -q \
    diffusers==0.27.2 \
    transformers==4.40.0 \
    accelerate==0.29.3 \
    peft==0.10.0 \
    wandb==0.16.6 \
    torchmetrics==1.3.2 \
    clean-fid==0.1.35 \
    lpips==0.1.4 \
    Pillow tqdm pyyaml numpy scipy

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()
login(token=secrets.get_secret('HF_TOKEN'))
wandb.login(key=secrets.get_secret('WANDB_API_KEY'))
print('HuggingFace: OK | Wandb: OK')

In [ ]:
import os
from pathlib import Path

WORK     = '/kaggle/working'
REPO     = f'{WORK}/defect-synthesis'     
CONFIG   = f'{REPO}/config.yaml'         
CKPT     = f'{WORK}/checkpoints'
SPLITS   = f'{WORK}/splits'
GENERATED = f'{WORK}/generated'
RESULTS  = f'{WORK}/results'


for d in [GENERATED, RESULTS, f'{WORK}/logs']:
    Path(d).mkdir(parents=True, exist_ok=True)

if not Path(REPO).exists():
    REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
    !git clone {REPO_URL} {REPO}

os.chdir(REPO)
print(f'Working dir: {os.getcwd()}')
print(f'Config:      {CONFIG}')
print(f'Checkpoints: {CKPT}')

In [ ]:
import json
from pathlib import Path
from safetensors.torch import load_file, save_file

def convert_checkpoint(old_dir: str, new_dir: str, force: bool = False):
    old_dir, new_dir = Path(old_dir), Path(new_dir)
    out_file = new_dir / 'pytorch_lora_weights.safetensors'

    if out_file.exists() and not force:
        print(f'  SKIP (already converted): {new_dir.name}')
        return True

    candidates = list(old_dir.glob('*.safetensors'))
    if not candidates:
        print(f'  No .safetensors found in {old_dir}')
        return False

    new_dir.mkdir(parents=True, exist_ok=True)
    sd = load_file(str(candidates[0]))

    new_sd = {}
    for k, v in sd.items():
        if k.startswith('base_model.model.'):
            new_key = 'unet.' + k[len('base_model.model.'):]
        elif not k.startswith('unet.'):
            new_key = 'unet.' + k
        else:
            new_key = k
        new_sd[new_key] = v

    save_file(new_sd, str(out_file))

    import torch
    norms = [v.float().norm().item() for v in new_sd.values()]
    avg_norm = sum(norms) / len(norms)
    print(f'  {old_dir.parent.name}/{old_dir.name} → {new_dir.name} | keys={len(new_sd)} | avg_norm={avg_norm:.4f}')
 
    return True


WORK = '/kaggle/working'
CKPT = f'{WORK}/checkpoints'


print('=== Stage 1 ===')
for cat in ['bottle', 'leather', 'metal_nut']:
    convert_checkpoint(
        old_dir=f'{CKPT}/stage1/{cat}/final',
        new_dir=f'{CKPT}/stage1/{cat}/final_fixed',
    )

print('\n=== Stage 2 ===')
category_defects = {
    'bottle':    ['broken_large', 'broken_small', 'contamination'],
    'leather':   ['color', 'cut', 'fold', 'glue', 'poke'],
    'metal_nut': ['bent', 'color', 'flip', 'scratch'],
}
for cat, defects in category_defects.items():
    for defect in defects:
        old = f'{CKPT}/stage2/{cat}/{defect}/final'
        new = f'{CKPT}/stage2/{cat}/{defect}/final_fixed'
        if Path(old).exists():
            convert_checkpoint(old_dir=old, new_dir=new)
        else:
            print(f'  no exists: stage2/{cat}/{defect}/final')

In [ ]:
from pathlib import Path

WORK = '/kaggle/working'
CKPT = f'{WORK}/checkpoints'

print('=== Stage 1 (final_fixed) ===')
for cat in ['bottle', 'leather', 'metal_nut']:
    p = Path(f'{CKPT}/stage1/{cat}/final_fixed/pytorch_lora_weights.safetensors')
    print(f'  {cat}: {"ok" if p.exists() else "missing"}')

print('\n=== Stage 2 (final_fixed) ===')
category_defects = {
    'bottle':    ['broken_large', 'broken_small', 'contamination'],
    'leather':   ['color', 'cut', 'fold', 'glue', 'poke'],
    'metal_nut': ['bent', 'color', 'flip', 'scratch'],
}
for cat, defects in category_defects.items():
    for defect in defects:
        p = Path(f'{CKPT}/stage2/{cat}/{defect}/final_fixed/pytorch_lora_weights.safetensors')
        print(f'  {cat}/{defect}: {"ok" if p.exists() else "missing"}')

In [ ]:
# Inference — two-stage pipeline (baseline n_shots=10) 
# Generate 8 images per category × defect

!python inference.py \
    --config config.yaml \
    --mode two_stage

gen_dir = Path(f'{WORK}/generated/two_stage')
runs = list(gen_dir.rglob('grid.png'))
print(f'\nGrids generated: {len(runs)}')
for r in runs:
    print(f'  {r.relative_to(gen_dir)}')

In [ ]:
# Visualize two-stage grids 
from PIL import Image as PILImage
from pathlib import Path
import IPython.display as display

CATEGORY    = 'bottle'
DEFECT_TYPE = 'broken_large'

grid_path = Path(f'{WORK}/generated/two_stage/{CATEGORY}/{DEFECT_TYPE}/baseline/grid.png')
if grid_path.exists():
    img = PILImage.open(grid_path)
    display.display(img)
    print(f'Grid: {grid_path}')
else:
    print(f'Grid not found: {grid_path}')

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
import json, time
from datetime import datetime

WORK = '/kaggle/working'
CKPT = f'{WORK}/checkpoints'

def make_grid(images, ncols=4, padding=8, bg=(30,30,46)):
    w, h = images[0].size
    nrows = (len(images) + ncols - 1) // ncols
    grid = Image.new('RGB',
        (ncols*w + (ncols+1)*padding, nrows*h + (nrows+1)*padding), bg)
    for idx, img in enumerate(images):
        r, c = divmod(idx, ncols)
        grid.paste(img, (padding + c*(w+padding), padding + r*(h+padding)))
    return grid

def add_caption(image, text, font_size=16):
    bar_h = font_size + 12
    out = Image.new('RGB', (image.width, image.height + bar_h), (30,30,46))
    out.paste(image, (0, bar_h))
    draw = ImageDraw.Draw(out)
    try:
        font = ImageFont.truetype(
            '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', font_size)
    except:
        font = ImageFont.load_default()
    draw.text((8, 6), text[:100], fill=(205,214,244), font=font)
    return out

def generate_two_stage(category, defect_type, n_shots=None,
                       weight_V=1.0, weight_D=0.8, num_images=8, seed=42):
   
    shots_tag  = f'shots{n_shots}' if n_shots else 'baseline'
    stage1_path = f'{CKPT}/stage1/{category}/final_fixed'
    stage2_path = f'{CKPT}/stage2/{category}/{defect_type}/final_fixed'
    out_dir     = Path(f'{WORK}/generated/two_stage/{category}/{defect_type}/{shots_tag}')

   
    for p, name in [(stage1_path, 'Stage1'), (stage2_path, 'Stage2')]:
        lora_file = Path(p) / 'pytorch_lora_weights.safetensors'
        if not lora_file.exists():
            print(f'  {name} missing: {p}')
            return None

    out_dir.mkdir(parents=True, exist_ok=True)


    pipe = StableDiffusionPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to('cuda')

    pipe.load_lora_weights(stage1_path, adapter_name='V')
    pipe.load_lora_weights(stage2_path, adapter_name='D')
    pipe.set_adapters(['V', 'D'], adapter_weights=[weight_V, weight_D])
    pipe.set_progress_bar_config(disable=True)

    prompt = f'a close-up of a sks {category} surface with a xjy anomaly, industrial inspection'
    negative = 'gun, rifle, weapon, knife, blade, firearm, bag, handbag, person, hand, logo, text'

    t0 = time.time()
    images = pipe(
        prompt,
        negative_prompt=negative,
        num_images_per_prompt=num_images,
        num_inference_steps=30,
        guidance_scale=7.5,
        generator=torch.Generator('cuda').manual_seed(seed),
    ).images
    elapsed = time.time() - t0

  
    for i, img in enumerate(images):
        img.save(out_dir / f'gen_{i:03d}.png')

    
    grid = make_grid(images, ncols=4)
    grid = add_caption(grid, f'{prompt[:80]}...')
    grid.save(out_dir / 'grid.png')

  
    meta = {
        'timestamp': datetime.now().isoformat(),
        'category': category, 'defect_type': defect_type,
        'n_shots': n_shots, 'shots_tag': shots_tag,
        'prompt': prompt, 'negative_prompt': negative,
        'weight_V': weight_V, 'weight_D': weight_D,
        'num_images': num_images, 'seed': seed,
        'generation_time_s': round(elapsed, 2),
        'stage1_path': stage1_path, 'stage2_path': stage2_path,
    }
    with open(out_dir / 'metadata.json', 'w') as f:
        json.dump(meta, f, indent=2)

    print(f'  ✓ {category}/{defect_type}/{shots_tag} — {num_images} img in {elapsed:.1f}s')

    del pipe
    torch.cuda.empty_cache()
    return images


In [ ]:
# Ablation study — n_shots = 5, 10, 20
# Generate for all shot values defined in config.yaml

!python inference.py \
    --config config.yaml \
    --mode two_stage \
    --ablation

print('\nAblation inference completed.')

In [ ]:
# Baseline — zero-shot inpainting
# SD 1.5 base without LoRA, generic prompt with defect name

!python inference.py \
    --config config.yaml \
    --mode zero_shot

print('\nZero-shot inference completed.')

In [ ]:
# Baseline — single-stage LoRA (training) 
# Before training the single-stage baseline for each category/defect

import json
from pathlib import Path

WORK = '/kaggle/working'
splits_dir = Path(f'{WORK}/splits')

for cat in ['bottle', 'metal_nut', 'leather']:
    split_file = splits_dir / f'{cat}_split.json'
    if not split_file.exists():
        print(f'Skip {cat} — split not found')
        continue
    with open(split_file) as f:
        split = json.load(f)
    for defect_type in split['stage2']:
        print(f'\n→ Training baseline single-stage: {cat}/{defect_type}')
        !python train/train_baseline_single.py \
            --config config.yaml \
            --category {cat} \
            --defect_type {defect_type} \
            --n_shots 10

print('\nBaseline single-stage training completed.')

In [ ]:
# Baseline — single-stage LoRA (inference)

!python inference.py \
    --config config.yaml \
    --mode single_stage

print('\nSingle-stage inference completed.')

In [ ]:
# Visual comparison grid 
# Show side by side: zero-shot | single-stage | two-stage
# for a specific category and defect type

from PIL import Image as PILImage, ImageDraw, ImageFont
from pathlib import Path
import IPython.display as ipd

WORK        = '/kaggle/working'
CATEGORY    = 'bottle'
DEFECT_TYPE = 'broken_large'
SHOTS_TAG   = 'baseline'    # or 'shots5', 'shots10', 'shots20'

def load_first_gen(mode):
    p = Path(f'{WORK}/generated/{mode}/{CATEGORY}/{DEFECT_TYPE}/{SHOTS_TAG}')
    pngs = sorted([x for x in p.glob('*.png') if x.name != 'grid.png'])
    if pngs:
        return PILImage.open(pngs[0]).convert('RGB').resize((256, 256))
    return PILImage.new('RGB', (256, 256), (60, 60, 60))

imgs   = [load_first_gen(m) for m in ['zero_shot', 'single_stage', 'two_stage']]
labels = ['Zero-Shot', 'Single-Stage LoRA', 'Two-Stage (ours)']

BAR_H = 30
PAD   = 8
W = len(imgs) * 256 + (len(imgs) + 1) * PAD
H = 256 + BAR_H + 2 * PAD

comparison = PILImage.new('RGB', (W, H), (30, 30, 46))
draw = ImageDraw.Draw(comparison)
try:
    font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 14)
except:
    font = ImageFont.load_default()

for i, (img, label) in enumerate(zip(imgs, labels)):
    x = PAD + i * (256 + PAD)
    comparison.paste(img, (x, BAR_H + PAD))
    draw.text((x + 5, 8), label, fill=(205, 214, 244), font=font)

comparison.save(f'{WORK}/results/comparison_{CATEGORY}_{DEFECT_TYPE}.png')
ipd.display(comparison)
print(f'Comparison: zero-shot vs single-stage vs two-stage — {CATEGORY}/{DEFECT_TYPE}')